In [1]:
# Import python modules
import os
import sys
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# Determine the absolute path to the src directory (one level up from notebooks)
module_path = os.path.abspath(os.path.join("..", "src"))
if module_path not in sys.path:
    sys.path.append(module_path)

project_root = os.path.abspath(os.path.join(".."))
if project_root not in sys.path:
    sys.path.insert(0, project_root)

In [2]:
# Now this works because Python sees `src` as a top-level module
from src import analytics, plotting, utils

In [3]:
import yaml

In [4]:
# TODO: Read both no_bat and bat results. Make a copy of the config with co2-prices

In [5]:
BASE_FOLDER = os.path.dirname(os.getcwd())
RUNS_FOLDER = os.path.join(BASE_FOLDER, "runs")
BATCH_RUNS_FOLDER = os.path.join(RUNS_FOLDER, "batch_runs")

DATA_FOLDER = os.path.join(BASE_FOLDER, "data")

In [6]:
FOLDER = r"C:\Users\tinus\OneDrive\Dokumenter\0 Master\code\master_project\runs\batch_runs\batch_128_4_4_4core_bat_and_no_bat\1_May16_Fri_h23_m07_s45-GTSEP_stochastic_v1-128_ES_PT"

In [10]:
decision_variables_folder = os.path.join(FOLDER, "decision_variables")
model_info_folder = os.path.join(FOLDER, "model_info")
dual_variables_folder = os.path.join(FOLDER, "dual_variables")

In [11]:
RESULTS_FOLDER = os.path.join(FOLDER, "results")

In [12]:
model_info = pd.read_csv(os.path.join(model_info_folder, "model_info.csv"))
config = yaml.safe_load(open(os.path.join(model_info_folder, "config.yaml")))
jsons = utils.read_jsons_from_dir(model_info_folder)

In [13]:
input_data_folder = os.path.join(
    os.path.dirname(os.getcwd()), "data", "processed", config["data_folder_name"]
)
input_data = utils.load_multi_year_csv_files_with_week_from_folder(
    years=config["years"], data_folder_path=input_data_folder, yearly_discount=100
)

In [14]:
generators = input_data["generators"]

In [15]:
input_data.keys()

dict_keys(['batteries', 'branches', 'capacity_factors', 'generators', 'generator_costs', 'hourly_demand', 'nodes'])

# Hourly demand by week analysis

In [16]:
hourly_demand = input_data["hourly_demand"]

In [17]:
import pandas as pd

# Assume hourly_demand is your dataframe and MultiIndex is ["week", "hour"]

# Sum across hours to get total demand for each week (sum across all nodes)
weekly_total = hourly_demand.sum(axis=1).groupby("week").sum()

In [18]:
def week_to_season(week):
    if 1 <= week <= 13:
        return "Winter"
    elif 14 <= week <= 26:
        return "Spring"
    elif 27 <= week <= 39:
        return "Summer"
    elif 40 <= week <= 52:
        return "Autumn"
    else:
        return "Unknown"


weekly_total = weekly_total.to_frame("total_demand")
weekly_total["season"] = weekly_total.index.map(week_to_season)

In [19]:
result = []
for season in ["Winter", "Spring", "Summer", "Autumn"]:
    season_weeks = weekly_total[weekly_total["season"] == season]
    if not season_weeks.empty:
        highest = season_weeks["total_demand"].idxmax()
        lowest = season_weeks["total_demand"].idxmin()
        result.append(
            {
                "season": season,
                "highest_week": highest,
                "highest_demand": season_weeks.loc[highest, "total_demand"],
                "lowest_week": lowest,
                "lowest_demand": season_weeks.loc[lowest, "total_demand"],
            }
        )

# Convert to DataFrame for pretty display
seasonal_extremes = pd.DataFrame(result)
print(seasonal_extremes)

   season  highest_week  highest_demand  lowest_week  lowest_demand
0  Winter             9    6.201472e+06           13   5.133926e+06
1  Spring            15    5.454663e+06           18   5.115533e+06
2  Summer            28    5.865407e+06           33   5.286365e+06
3  Autumn            48    6.217299e+06           44   5.136201e+06


In [20]:
# 1. Aggregate total demand by week
weekly_total = hourly_demand.sum(axis=1).groupby("week").sum()


# 2. Assign seasons
def week_to_season(week):
    if 1 <= week <= 13:
        return "Winter"
    elif 14 <= week <= 26:
        return "Spring"
    elif 27 <= week <= 39:
        return "Summer"
    elif 40 <= week <= 52:
        return "Autumn"
    else:
        return "Unknown"


weekly_total = weekly_total.to_frame("total_demand")
weekly_total["season"] = weekly_total.index.map(week_to_season)

# 3. Sort weeks in each season by demand descending
for season in ["Winter", "Spring", "Summer", "Autumn"]:
    season_weeks = weekly_total[weekly_total["season"] == season]
    sorted_weeks = season_weeks.sort_values("total_demand", ascending=False)
    print(f"\nSeason: {season}")
    print(sorted_weeks[["total_demand"]])


Season: Winter
      total_demand
week              
9     6.201472e+06
4     6.169147e+06
3     6.154159e+06
1     6.152327e+06
6     6.090839e+06
7     5.999782e+06
2     5.986252e+06
8     5.964497e+06
5     5.921483e+06
11    5.909982e+06
10    5.855242e+06
12    5.702891e+06
13    5.133926e+06

Season: Spring
      total_demand
week              
15    5.454663e+06
24    5.409832e+06
26    5.393359e+06
14    5.379059e+06
25    5.298752e+06
23    5.235780e+06
21    5.218462e+06
16    5.209588e+06
17    5.204575e+06
20    5.203051e+06
19    5.200285e+06
22    5.189140e+06
18    5.115533e+06

Season: Summer
      total_demand
week              
28    5.865407e+06
29    5.739834e+06
30    5.716870e+06
27    5.692057e+06
31    5.577737e+06
34    5.544469e+06
36    5.495223e+06
38    5.489905e+06
39    5.445019e+06
32    5.429892e+06
37    5.376055e+06
35    5.338719e+06
33    5.286365e+06

Season: Autumn
      total_demand
week              
48    6.217299e+06
50    6.131717e+06
49   

In [21]:
# 1. Aggregate total demand by week
weekly_total = hourly_demand.sum(axis=1).groupby("week").sum()


# 2. Assign seasons
def week_to_season(week):
    if 1 <= week <= 13:
        return "Winter"
    elif 14 <= week <= 26:
        return "Spring"
    elif 27 <= week <= 39:
        return "Summer"
    elif 40 <= week <= 52:
        return "Autumn"
    else:
        return "Unknown"


weekly_total = weekly_total.to_frame("total_demand")
weekly_total["season"] = weekly_total.index.map(week_to_season)

# 3. Calculate extremes and summary for each season
summary_rows = []
for season in ["Winter", "Spring", "Summer", "Autumn"]:
    season_weeks = weekly_total[weekly_total["season"] == season]
    if season_weeks.empty:
        continue
    highest_week = season_weeks["total_demand"].idxmax()
    highest_val = season_weeks.loc[highest_week, "total_demand"]
    lowest_week = season_weeks["total_demand"].idxmin()
    lowest_val = season_weeks.loc[lowest_week, "total_demand"]
    mean_val = season_weeks["total_demand"].mean()
    diff = highest_val - lowest_val
    pct_diff = 100 * diff / lowest_val if lowest_val != 0 else float("inf")
    summary_rows.append(
        {
            "season": season,
            "highest_week": highest_week,
            "highest_demand": highest_val,
            "lowest_week": lowest_week,
            "lowest_demand": lowest_val,
            "mean_weekly_demand": mean_val,
            "difference": diff,
            "percent_difference": pct_diff,
        }
    )

summary_df = pd.DataFrame(summary_rows)
summary_df

,season,highest_week,highest_demand,lowest_week,lowest_demand,mean_weekly_demand,difference,percent_difference
0,Winter,9,6.201472e+06,13,5.133926e+06,5.941692e+06,1.067545e+06,20.793937
1,Spring,15,5.454663e+06,18,5.115533e+06,5.270160e+06,3.391295e+05,6.629406
2,Summer,28,5.865407e+06,33,5.286365e+06,5.538273e+06,5.790428e+05,10.953516
3,Autumn,48,6.217299e+06,44,5.136201e+06,5.578973e+06,1.081098e+06,21.048596


# Renewable mapping (is generator renewable or not)

In [22]:
# Initialize mapping
is_renewable = {}
generators = input_data["generators"]

In [23]:
capacity_factors = input_data["capacity_factors"]
G = generators.index.get_level_values("generator").unique().tolist()
# Initialize mapping
is_renewable = {}


def _make_is_renewable_mapping(generator_list, capacity_factors_df):
    """
    Returns a mapping dictionary: {generator: True (renewable) or False (non-renewable)}
    A generator is considered non-renewable if its capacity factor is 1.0 for all time steps.
    Otherwise, it is considered renewable.

    Args:
        generator_list (list): List of generator names (str).
        capacity_factors_df (pd.DataFrame): DataFrame with generators as columns.

    Returns:
        dict: {generator: True/False}
    """
    mapping = {}
    for gen in generator_list:
        if gen not in capacity_factors_df.columns:
            mapping[gen] = np.nan  # or False, or raise an error
        else:
            vals = capacity_factors_df[gen].values
            if np.allclose(vals, 1.0, atol=1e-8):
                mapping[gen] = False  # Non-renewable
            else:
                mapping[gen] = True  # Renewable
    return mapping


is_renewable = _make_is_renewable_mapping(G, capacity_factors)
print(is_renewable)


# Print (or use) the mapping
for gen, renew in is_renewable.items():
    print(f"{gen}: {'renewable' if renew else 'non-renewable'}")
    print(is_renewable[gen])

{'ES1 0 CCGT': False, 'ES1 0 coal': False, 'ES1 0 offwind-ac': True, 'ES1 0 onwind': True, 'ES1 0 solar': True, 'ES1 1 CCGT': False, 'ES1 1 coal': False, 'ES1 1 offwind-ac': True, 'ES1 1 onwind': True, 'ES1 1 solar': True, 'ES1 2 CCGT': False, 'ES1 2 coal': False, 'ES1 2 offwind-ac': True, 'ES1 2 onwind': True, 'ES1 2 ror': True, 'ES1 2 solar': True, 'ES1 3 CCGT': False, 'ES1 3 coal': False, 'ES1 3 offwind-ac': True, 'ES1 3 onwind': True, 'ES1 3 solar': True, 'ES1 4 CCGT': False, 'ES1 4 coal': False, 'ES1 4 offwind-ac': True, 'ES1 4 onwind': True, 'ES1 4 solar': True, 'ES1 5 CCGT': False, 'ES1 5 onwind': True, 'ES1 5 solar': True, 'ES1 6 CCGT': False, 'ES1 6 offwind-ac': True, 'ES1 6 onwind': True, 'ES1 6 solar': True, 'ES1 7 CCGT': False, 'ES1 7 onwind': True, 'ES1 7 solar': True, 'ES1 8 CCGT': False, 'ES1 8 offwind-ac': True, 'ES1 8 onwind': True, 'ES1 8 ror': True, 'ES1 8 solar': True, 'PT1 0 CCGT': False, 'PT1 0 offwind-ac': True, 'PT1 0 onwind': True, 'PT1 0 ror': True, 'PT1 0 sol

# Checking if asset prices changes by year

In [24]:
generators = input_data["generators"]
branches = input_data["branches"]
batteries = input_data["batteries"]

In [25]:
generators.groupby("year")["capital_cost"].mean()

year
2025    140040.843194
2030    139940.843194
2040    139840.843194
2050    139740.843194
Name: capital_cost, dtype: float64

In [26]:
branches.groupby("year")["capital_cost"].mean()

year
2025    17814.09335
2030    17714.09335
2040    17614.09335
2050    17514.09335
Name: capital_cost, dtype: float64

In [27]:
batteries.groupby("year")["capital_cost"].mean()

year
2025    21958.92494
2030    21858.92494
2040    21758.92494
2050    21658.92494
Name: capital_cost, dtype: float64

In [28]:
num_hours = [i for i in range(1, 25)]
for i, hours in enumerate(num_hours):
    print(hours)
    print(f"100% ramp-up in {hours} hours equals: {1 / hours * 100:.2f}%")

1
100% ramp-up in 1 hours equals: 100.00%
2
100% ramp-up in 2 hours equals: 50.00%
3
100% ramp-up in 3 hours equals: 33.33%
4
100% ramp-up in 4 hours equals: 25.00%
5
100% ramp-up in 5 hours equals: 20.00%
6
100% ramp-up in 6 hours equals: 16.67%
7
100% ramp-up in 7 hours equals: 14.29%
8
100% ramp-up in 8 hours equals: 12.50%
9
100% ramp-up in 9 hours equals: 11.11%
10
100% ramp-up in 10 hours equals: 10.00%
11
100% ramp-up in 11 hours equals: 9.09%
12
100% ramp-up in 12 hours equals: 8.33%
13
100% ramp-up in 13 hours equals: 7.69%
14
100% ramp-up in 14 hours equals: 7.14%
15
100% ramp-up in 15 hours equals: 6.67%
16
100% ramp-up in 16 hours equals: 6.25%
17
100% ramp-up in 17 hours equals: 5.88%
18
100% ramp-up in 18 hours equals: 5.56%
19
100% ramp-up in 19 hours equals: 5.26%
20
100% ramp-up in 20 hours equals: 5.00%
21
100% ramp-up in 21 hours equals: 4.76%
22
100% ramp-up in 22 hours equals: 4.55%
23
100% ramp-up in 23 hours equals: 4.35%
24
100% ramp-up in 24 hours equals: 4.17%

4 hour ramp max for CCGT equals 0.25 ramp rate
2 hour ramp max for CCGT equals 0.5 ramp rate

16 hour ramp max for coal equals 0.0625 ramp rate
8 hour ramp max for coal equals 0.125 ramp rate

Ratio ramp rate_CCGT = 4x ramp rate_coal

In [31]:
for g in G:
    print(g)
    print("CCGT" in g)
    print("coal" in g)
    print("---")

ES1 0 CCGT
True
False
---
ES1 0 coal
False
True
---
ES1 0 offwind-ac
False
False
---
ES1 0 onwind
False
False
---
ES1 0 solar
False
False
---
ES1 1 CCGT
True
False
---
ES1 1 coal
False
True
---
ES1 1 offwind-ac
False
False
---
ES1 1 onwind
False
False
---
ES1 1 solar
False
False
---
ES1 2 CCGT
True
False
---
ES1 2 coal
False
True
---
ES1 2 offwind-ac
False
False
---
ES1 2 onwind
False
False
---
ES1 2 ror
False
False
---
ES1 2 solar
False
False
---
ES1 3 CCGT
True
False
---
ES1 3 coal
False
True
---
ES1 3 offwind-ac
False
False
---
ES1 3 onwind
False
False
---
ES1 3 solar
False
False
---
ES1 4 CCGT
True
False
---
ES1 4 coal
False
True
---
ES1 4 offwind-ac
False
False
---
ES1 4 onwind
False
False
---
ES1 4 solar
False
False
---
ES1 5 CCGT
True
False
---
ES1 5 onwind
False
False
---
ES1 5 solar
False
False
---
ES1 6 CCGT
True
False
---
ES1 6 offwind-ac
False
False
---
ES1 6 onwind
False
False
---
ES1 6 solar
False
False
---
ES1 7 CCGT
True
False
---
ES1 7 onwind
False
False
---
ES1 7 sola

In [7]:
path = r"C:\Users\tinus\OneDrive\Dokumenter\0 Master\code\master_project\runs\batch_runs\batch_128_sv2_8_weeks_ef05_rf025_bnb_sens_co2price"

In [8]:
os.listdir(path)

['10_May24_Sat_h02_m19_s08-GTSEP_stochastic_v2-128_ES_PT_sv2_no_bat',
 '1_May23_Fri_h21_m17_s38-GTSEP_stochastic_v2-128_ES_PT_sv2',
 '2_May23_Fri_h22_m05_s43-GTSEP_stochastic_v2-128_ES_PT_sv2',
 '3_May23_Fri_h22_m43_s58-GTSEP_stochastic_v2-128_ES_PT_sv2',
 '4_May23_Fri_h23_m32_s57-GTSEP_stochastic_v2-128_ES_PT_sv2',
 '5_May24_Sat_h00_m10_s28-GTSEP_stochastic_v2-128_ES_PT_sv2',
 '6_May24_Sat_h00_m56_s01-GTSEP_stochastic_v2-128_ES_PT_sv2_no_bat',
 '7_May24_Sat_h01_m17_s36-GTSEP_stochastic_v2-128_ES_PT_sv2_no_bat',
 '8_May24_Sat_h01_m35_s44-GTSEP_stochastic_v2-128_ES_PT_sv2_no_bat',
 '9_May24_Sat_h01_m57_s43-GTSEP_stochastic_v2-128_ES_PT_sv2_no_bat']

In [10]:
import os

path = r"C:\Users\tinus\OneDrive\Dokumenter\0 Master\code\master_project\runs\batch_runs\batch_128_sv2_8_weeks_ef05_rf025_bnb_sens_co2price"

for folder in os.listdir(path):
    full_old = os.path.join(path, folder)
    if os.path.isdir(full_old):
        # Keep the first number before the first underscore
        parts = folder.split("_", 1)
        if len(parts) == 2:
            number = parts[0]
            rest = parts[1]
            # Remove everything up to and including the first dash in the rest
            if "-" in rest:
                rest = rest.split("-", 1)[1]
            newname = f"{number}_{rest}"
        else:
            newname = folder
        # Replace 'GTSEP_stochastic_v2' with 'sv2'
        newname = newname.replace("GTSEP_stochastic_v2", "sv2")
        full_new = os.path.join(path, newname)
        print(f"Renaming:\n  {full_old}\n  --> {full_new}")
        os.rename(full_old, full_new)

Renaming:
  C:\Users\tinus\OneDrive\Dokumenter\0 Master\code\master_project\runs\batch_runs\batch_128_sv2_8_weeks_ef05_rf025_bnb_sens_co2price\10_May24_Sat_h02_m19_s08-GTSEP_stochastic_v2-128_ES_PT_sv2_no_bat
  --> C:\Users\tinus\OneDrive\Dokumenter\0 Master\code\master_project\runs\batch_runs\batch_128_sv2_8_weeks_ef05_rf025_bnb_sens_co2price\10_sv2-128_ES_PT_sv2_no_bat
Renaming:
  C:\Users\tinus\OneDrive\Dokumenter\0 Master\code\master_project\runs\batch_runs\batch_128_sv2_8_weeks_ef05_rf025_bnb_sens_co2price\1_May23_Fri_h21_m17_s38-GTSEP_stochastic_v2-128_ES_PT_sv2
  --> C:\Users\tinus\OneDrive\Dokumenter\0 Master\code\master_project\runs\batch_runs\batch_128_sv2_8_weeks_ef05_rf025_bnb_sens_co2price\1_sv2-128_ES_PT_sv2
Renaming:
  C:\Users\tinus\OneDrive\Dokumenter\0 Master\code\master_project\runs\batch_runs\batch_128_sv2_8_weeks_ef05_rf025_bnb_sens_co2price\2_May23_Fri_h22_m05_s43-GTSEP_stochastic_v2-128_ES_PT_sv2
  --> C:\Users\tinus\OneDrive\Dokumenter\0 Master\code\master_proje